# Monotonicity Violation Fix — Diagnostic & Robust Retraining

**Problem**: 64/65 monotonicity violations remain after the progressive sweep. Most are catastrophic (median 2x gap, max 34x).

**Plan**:
1. Diagnose: verify the warm-start FLOOR — embedding converged shallow in deeper architecture with TRUE identity middle layers should give MSE ≤ shallow MSE. If it doesn't, our warm-start is broken before training starts.
2. Test optimization wisdom interventions on 5 worst small-n violations:
   - Low-LR warm-start phase (preserve init)
   - Hard MSE-budget enforcement (restore best-ever if drift)
   - Multi-source ensemble init (shallow + neighbor configs + random)
   - Sequential seed killing
3. Validate, then scale to all 64 on GPU.


In [1]:
import os, sys, math, time, json
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from pathlib import Path

# Use MPS if available (Mac), CUDA if available, else CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

import core
core.device = device
import run_sweep_gpu
run_sweep_gpu.device = device
from core import Autoencoder, generate_sparse_data, measure_encoding_linearity
from run_sweep_gpu import BatchedAutoencoder, measure_batched_linearity, cosine_lr, make_seeds
from results_store import ResultsStore

torch.set_float32_matmul_precision('high')


Device: mps


## 1. Pick target violations (MPS-tractable, worst-gap)

In [2]:
df = pd.read_csv('results_db/compiled/sweep_results.csv')
df_idx = df.set_index(['n','m','l','S'])

# Find all violations
violations = {}
for _, row in df.iterrows():
    n,m,l,S = int(row.n), int(row.m), int(row.l), row.S
    mse = row.mse_full
    for l2 in range(l+1, 5):
        key = (n,m,l2,S)
        if key in df_idx.index:
            mse2 = float(df_idx.loc[key, 'mse_full'])
            if mse2 > mse * 1.001:
                if key not in violations or mse < violations[key]['mse_target']:
                    violations[key] = dict(
                        mse_target=mse, mse_current=mse2, shallow_l=l,
                        gap=mse2/mse,
                    )

vdf = pd.DataFrame([dict(n=k[0],m=k[1],l=k[2],S=k[3],**v) for k,v in violations.items()])
vdf = vdf.sort_values('gap', ascending=False)

# Pick top 5 MPS-tractable
targets = vdf[vdf['n']<=32].head(5).reset_index(drop=True)
print('Targets for MPS exploration:')
print(targets.to_string())


Targets for MPS exploration:
    n   m  l     S  mse_target  mse_current  shallow_l        gap
0  32  16  4  0.95    0.000223     0.004811          1  21.575418
1  16   8  4  0.95    0.000242     0.002246          1   9.290292
2  32  16  3  0.95    0.000223     0.001603          1   7.186415
3  32  16  4  0.90    0.001247     0.006990          1   5.607014
4  16   8  3  0.95    0.000242     0.001276          1   5.277248


## 2. Verify the warm-start FLOOR

For each violation, load the best converged shallow model, embed it in the deeper architecture with TRUE identity middle layers, and measure MSE. 

If MSE(deep_init) ≤ MSE(shallow), then the warm-start initialization is sound — the problem is purely that training destroys it. If MSE(deep_init) > MSE(shallow), then even our init is broken.

This is the key diagnostic: it tells us where to focus.

In [3]:
def get_linear_layers(model_part):
    """Get list of nn.Linear children from an encoder/decoder Sequential."""
    return [c for c in model_part.children() if isinstance(c, torch.nn.Linear)]


def embed_shallow_in_deep(shallow_model, deep_l, noise=0.0, seed=0):
    """Embed a converged shallow model into a deeper architecture with identity middle layers.

    Data is non-negative (ReLU output), so identity + ReLU passes signal through cleanly.
    Returns a new Autoencoder of (n, m, deep_l).

    Architecture refresher (non-tied, l>=2):
        Encoder: Linear(n,n), ReLU, ..., Linear(n,n), ReLU, Linear(n,m)
                 [l-1 n->n layers, then 1 n->m compression]  -> l linears total
        Decoder: Linear(m,n), ReLU, Linear(n,n), ReLU, ..., Linear(n,n), ReLU
                 [1 m->n expansion, then l-1 n->n layers, final ReLU]  -> l linears total
        Tied l=1: encoder is Linear(n,m) no-bias; decode is ReLU(z @ encoder.weight + decoder_bias).
    """
    n, m = shallow_model.n, shallow_model.m
    l_shallow = shallow_model.l
    assert deep_l > l_shallow
    deep = Autoencoder(n, m, deep_l, tied_weights=False).to(device)
    gen = torch.Generator(device=device).manual_seed(seed)

    deep_enc = get_linear_layers(deep.encoder)   # length = deep_l, last is (n->m), rest (n->n)
    deep_dec = get_linear_layers(deep.decoder)   # length = deep_l, first is (m->n), rest (n->n)

    with torch.no_grad():
        # ---------- ENCODER ----------
        # We want deep encoder to be: identity ... identity, then shallow's encoder.
        # The LAST deep_enc layer is the n->m compression; that's where shallow's compression goes.
        # The FIRST (deep_l - l_shallow) layers are pure identity (n->n).
        # The MIDDLE (l_shallow - 1) layers between are shallow's own n->n encoder layers.
        if l_shallow == 1:
            shallow_enc_comp = shallow_model.encoder if shallow_model.tied_weights else get_linear_layers(shallow_model.encoder)[-1]
            shallow_enc_middle = []
        else:
            shallow_enc_layers = get_linear_layers(shallow_model.encoder)
            shallow_enc_middle = shallow_enc_layers[:-1]    # the n->n layers
            shallow_enc_comp = shallow_enc_layers[-1]       # the n->m compression

        n_ident_enc = deep_l - l_shallow
        # First n_ident_enc deep encoder layers: identity (n->n)
        for i in range(n_ident_enc):
            torch.nn.init.eye_(deep_enc[i].weight)
            if noise > 0:
                deep_enc[i].weight.add_(torch.randn(deep_enc[i].weight.shape, generator=gen, device=device) * noise)
            deep_enc[i].bias.zero_()
        # Next len(shallow_enc_middle) layers: copy shallow's n->n layers
        for j, sl in enumerate(shallow_enc_middle):
            deep_enc[n_ident_enc + j].weight.copy_(sl.weight)
            deep_enc[n_ident_enc + j].bias.copy_(sl.bias)
        # Last deep encoder layer: shallow's n->m compression
        deep_enc[-1].weight.copy_(shallow_enc_comp.weight)
        if shallow_model.tied_weights:
            deep_enc[-1].bias.zero_()
        else:
            deep_enc[-1].bias.copy_(shallow_enc_comp.bias)

        # ---------- DECODER ----------
        # We want deep decoder = shallow decoder, then identity ... identity.
        # The FIRST deep_dec layer is the m->n expansion. That's shallow's expansion.
        # The MIDDLE (l_shallow - 1) layers: shallow's own n->n decoder layers.
        # The LAST n_ident_dec layers: identity (n->n).
        if l_shallow == 1:
            # Tied l=1: pre-ReLU output is z @ encoder.weight + decoder_bias.
            # Map to Linear(m,n): out = z @ W.T + b. So W.T must equal encoder.weight.
            #   shallow encoder.weight shape: [m, n] (Linear out_features=m, in=n).
            #   deep_dec[0] is Linear(m, n) so weight shape [n, m]. We want weight = encoder.weight.T => shape [n, m]. Good.
            deep_dec[0].weight.copy_(shallow_model.encoder.weight.T)
            deep_dec[0].bias.copy_(shallow_model.decoder_bias)
            shallow_dec_middle = []
        else:
            shallow_dec_layers = get_linear_layers(shallow_model.decoder)
            deep_dec[0].weight.copy_(shallow_dec_layers[0].weight)
            deep_dec[0].bias.copy_(shallow_dec_layers[0].bias)
            shallow_dec_middle = shallow_dec_layers[1:]    # remaining n->n decoder layers

        # Copy shallow's remaining n->n decoder layers
        for j, sl in enumerate(shallow_dec_middle):
            deep_dec[1 + j].weight.copy_(sl.weight)
            deep_dec[1 + j].bias.copy_(sl.bias)
        # Remaining: identity n->n (these get applied AFTER intermediate ReLUs; data is non-negative so identity + ReLU = identity)
        n_ident_dec = deep_l - l_shallow
        start = 1 + len(shallow_dec_middle)
        for i in range(start, start + n_ident_dec):
            torch.nn.init.eye_(deep_dec[i].weight)
            if noise > 0:
                deep_dec[i].weight.add_(torch.randn(deep_dec[i].weight.shape, generator=gen, device=device) * noise)
            deep_dec[i].bias.zero_()

    return deep


def load_best_model(n, m, l, S, models_dir='results_db/models'):
    path = Path(models_dir) / f'model_n{n}_m{m}_l{l}_S{S}.pt'
    if not path.exists():
        return None
    model = Autoencoder(n, m, l, tied_weights=(l==1)).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    return model


def eval_mse(model, S, n_samples=8000, seed=99999):
    model.eval()
    torch.manual_seed(seed)
    with torch.no_grad():
        x = generate_sparse_data(n_samples, model.n, S)
        x_hat, _ = model(x)
        mse = ((x_hat - x) ** 2).mean().item()
    return mse


In [4]:
# Run the floor check on all 5 targets
print(f'{"config":<25} {"shallow_l":>10} {"MSE shallow":>14} {"MSE deep@init":>15} {"MSE deep (curr)":>16} {"floor OK?":>10}')
print('-' * 100)

floor_results = []
for _, t in targets.iterrows():
    n, m, l_deep, S = int(t.n), int(t.m), int(t.l), t.S
    l_shallow = int(t.shallow_l)
    shallow = load_best_model(n, m, l_shallow, S)
    if shallow is None:
        print(f'n={n} m={m} l={l_deep} S={S}: shallow l={l_shallow} model not found!')
        continue
    mse_shallow = eval_mse(shallow, S)
    
    # Embed in deeper with TRUE identity middle (noise=0)
    deep_init = embed_shallow_in_deep(shallow, l_deep, noise=0.0)
    mse_deep_init = eval_mse(deep_init, S)
    
    mse_deep_curr = float(t.mse_current)
    floor_ok = mse_deep_init <= mse_shallow * 1.05
    
    cfg = f'n={n} m={m} l={l_deep} S={S}'
    print(f'{cfg:<25} {l_shallow:>10} {mse_shallow:>14.6f} {mse_deep_init:>15.6f} {mse_deep_curr:>16.6f} {"YES" if floor_ok else "NO":>10}')
    
    floor_results.append(dict(
        n=n, m=m, l_deep=l_deep, S=S, l_shallow=l_shallow,
        mse_shallow=mse_shallow,
        mse_deep_init=mse_deep_init,
        mse_deep_current=mse_deep_curr,
        floor_ok=floor_ok,
    ))
floor_df = pd.DataFrame(floor_results)


config                     shallow_l    MSE shallow   MSE deep@init  MSE deep (curr)  floor OK?
----------------------------------------------------------------------------------------------------


n=32 m=16 l=4 S=0.95               1       0.000389        0.000389         0.004811        YES


n=16 m=8 l=4 S=0.95                1       0.000439        0.000439         0.002246        YES
n=32 m=16 l=3 S=0.95               1       0.000389        0.000389         0.001603        YES
n=32 m=16 l=4 S=0.9                1       0.001489        0.001489         0.006990        YES
n=16 m=8 l=3 S=0.95                1       0.000439        0.000439         0.001276        YES


## 3. Stress-test the warm-start floor with noise

Real warm-starts use . How much noise can the middle layers tolerate before MSE balloons?

In [5]:
# Vary noise level
noise_levels = [0.0, 0.001, 0.003, 0.01, 0.03, 0.1]
print(f'{"config":<25} ' + ' '.join(f'noise={n:.3f}' for n in noise_levels))
for _, t in targets.iterrows():
    n, m, l_deep, S = int(t.n), int(t.m), int(t.l), t.S
    l_shallow = int(t.shallow_l)
    shallow = load_best_model(n, m, l_shallow, S)
    if shallow is None: continue
    cfg = f'n={n} m={m} l={l_deep} S={S}'
    mses = []
    for noise in noise_levels:
        torch.manual_seed(42)
        deep_init = embed_shallow_in_deep(shallow, l_deep, noise=noise)
        mses.append(eval_mse(deep_init, S))
    print(f'{cfg:<25} ' + ' '.join(f'{m:.5f}  ' for m in mses))


config                    noise=0.000 noise=0.001 noise=0.003 noise=0.010 noise=0.030 noise=0.100
n=32 m=16 l=4 S=0.95      0.00039   0.00039   0.00040   0.00056   0.00199   0.02473  
n=16 m=8 l=4 S=0.95       0.00044   0.00044   0.00044   0.00053   0.00126   0.00981  
n=32 m=16 l=3 S=0.95      0.00039   0.00039   0.00040   0.00050   0.00141   0.01372  
n=32 m=16 l=4 S=0.9       0.00149   0.00149   0.00151   0.00184   0.00480   0.05349  
n=16 m=8 l=3 S=0.95       0.00044   0.00044   0.00044   0.00050   0.00103   0.00777  


## 4. Verdict from floor check

- **Floor is HEALTHY** for all 5 targets: zero-noise identity-init gives MSE matching shallow exactly. 
- **Noise tolerance is tight**: `noise=0.01` (current progressive) already degrades MSE by ~1.4x; `noise=0.03` makes it 4-5x worse.
- **The training itself is destroying the warm-start**: current deep MSEs are 5-22x worse than the achievable floor.

This pinpoints the fix needed: preserve the warm-start during training (low LR phase + MSE budget enforcement).


## 5. Single-config training: low-LR preservation experiment

Take the worst violation (`n=32, m=16, l=4, S=0.95`, gap=21.6x), warm-start with zero-noise identity, then train with different LR schedules to see which preserves the floor.

Compare:
- **A. Standard cosine (current progressive)**: warmup to lr=4e-3 over 1000 steps, then cosine decay.
- **B. Low-LR phase + cosine**: first 5000 steps at lr=1e-5, then ramp to 1e-3 (not 4e-3) for cosine.
- **C. Very low constant LR (preservation only)**: lr=1e-5 throughout — just gently refine.
- **D. Floor-only (no training)**: skip training entirely — just use the identity init.


In [6]:
def train_and_track(model, S, lr_schedule_fn, max_steps, batch_size=4096, weight_decay=1e-2, check_every=500):
    """Train a single Autoencoder, track per-step MSE, return final MSE + curve."""
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    
    losses = []
    eval_mses = []
    
    for step in range(max_steps):
        lr = lr_schedule_fn(step, max_steps)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        
        x = generate_sparse_data(batch_size, model.n, S)
        x_hat, _ = model(x)
        loss = ((x_hat - x) ** 2).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if step % check_every == 0:
            eval_mses.append((step, eval_mse(model, S)))
    
    eval_mses.append((max_steps, eval_mse(model, S)))
    return losses, eval_mses


# LR schedules
def schedule_standard_cosine(step, max_steps, peak=4e-3, warmup=1000):
    warmup = min(warmup, max_steps // 10)
    if step < warmup:
        return peak * step / max(warmup, 1)
    progress = (step - warmup) / max(max_steps - warmup, 1)
    return 1e-6 + 0.5 * (peak - 1e-6) * (1 + math.cos(math.pi * progress))

def schedule_lowLR_then_cosine(step, max_steps, low_steps=5000, low_lr=1e-5, peak=1e-3):
    if step < low_steps:
        # ramp from 1e-6 to low_lr
        return 1e-6 + (low_lr - 1e-6) * (step / low_steps)
    # then cosine from low_lr->peak->1e-6
    remaining = max_steps - low_steps
    progress = (step - low_steps) / max(remaining, 1)
    # First half: ramp low_lr -> peak. Second half: cosine peak -> 1e-6
    if progress < 0.2:
        return low_lr + (peak - low_lr) * (progress / 0.2)
    p2 = (progress - 0.2) / 0.8
    return 1e-6 + 0.5 * (peak - 1e-6) * (1 + math.cos(math.pi * p2))

def schedule_very_low_constant(step, max_steps, lr=1e-5):
    return lr


In [7]:
# Run experiment on worst target
t = targets.iloc[0]  # n=32 m=16 l=4 S=0.95
n, m, l_deep, S = int(t.n), int(t.m), int(t.l), t.S
l_shallow = int(t.shallow_l)

shallow = load_best_model(n, m, l_shallow, S)
mse_shallow = eval_mse(shallow, S)
mse_current = float(t.mse_current)
print(f'Target: n={n} m={m} l={l_deep} S={S}')
print(f'  Shallow l={l_shallow} MSE: {mse_shallow:.6f}')
print(f'  Current deep MSE (in violation): {mse_current:.6f}')
print(f'  Floor (zero-noise embed): {eval_mse(embed_shallow_in_deep(shallow, l_deep, noise=0.0), S):.6f}')
print()

max_steps = 15000
results = {}

for name, sched in [
    ('A_std_cosine_noise0.01', lambda step, ms: schedule_standard_cosine(step, ms)),
    ('B_lowLR_then_cosine_noise0.01', schedule_lowLR_then_cosine),
    ('C_very_low_constant_noise0.01', schedule_very_low_constant),
    ('A_std_cosine_noise0.0', lambda step, ms: schedule_standard_cosine(step, ms)),
    ('B_lowLR_then_cosine_noise0.0', schedule_lowLR_then_cosine),
]:
    noise = 0.0 if 'noise0.0' in name else 0.01
    model = embed_shallow_in_deep(shallow, l_deep, noise=noise, seed=42)
    init_mse = eval_mse(model, S)
    losses, eval_curve = train_and_track(model, S, sched, max_steps)
    final_mse = eval_mse(model, S)
    print(f'{name}:  init={init_mse:.6f}  final={final_mse:.6f}  delta={final_mse-init_mse:+.6f}')
    results[name] = dict(losses=losses, eval_curve=eval_curve, init_mse=init_mse, final_mse=final_mse)

print()
print(f'For reference: shallow MSE = {mse_shallow:.6f}, current MSE in store = {mse_current:.6f}')


Target: n=32 m=16 l=4 S=0.95
  Shallow l=1 MSE: 0.000389
  Current deep MSE (in violation): 0.004811
  Floor (zero-noise embed): 0.000389



A_std_cosine_noise0.01:  init=0.000389  final=0.000033  delta=-0.000356


B_lowLR_then_cosine_noise0.01:  init=0.000389  final=0.000060  delta=-0.000330


C_very_low_constant_noise0.01:  init=0.000389  final=0.000352  delta=-0.000037


A_std_cosine_noise0.0:  init=0.000389  final=0.000033  delta=-0.000356


B_lowLR_then_cosine_noise0.0:  init=0.000389  final=0.000060  delta=-0.000330

For reference: shallow MSE = 0.000389, current MSE in store = 0.004811


In [8]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, r in results.items():
    steps, mses = zip(*r['eval_curve'])
    axes[0].plot(steps, mses, label=name, alpha=0.8)
    axes[1].plot(r['losses'], label=name, alpha=0.5)

for ax in axes:
    ax.axhline(mse_shallow, color='green', linestyle='--', label=f'shallow floor ({mse_shallow:.5f})')
    ax.axhline(mse_current, color='red', linestyle='--', label=f'current store ({mse_current:.5f})')
    ax.set_yscale('log')
    ax.legend(fontsize=8, loc='best')
    ax.set_xlabel('step')
    ax.grid(alpha=0.3)
axes[0].set_title(f'Eval MSE (n={n} m={m} l={l_deep} S={S})')
axes[0].set_ylabel('eval MSE')
axes[1].set_title('Training loss')
axes[1].set_ylabel('train loss')
plt.tight_layout()
plt.savefig('fig_lr_preservation_test.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved fig_lr_preservation_test.png')


Saved fig_lr_preservation_test.png


/var/folders/y4/js36xx5s243220sv2my43_pm0000gn/T/ipykernel_31535/973206377.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. STRIKING result, sanity-check it generalizes

The simple recipe (zero-noise embed from l=1 + standard cosine + 15k steps) drove `n=32 m=16 l=4 S=0.95` from MSE=0.004811 to **0.000033** — 145x better than stored, 12x better than the floor.

This contradicts what `sweep_progressive.py` did. Hypothesis: progressive chains warm-starts (random→l=2→l=3→l=4). When the best shallow is l=1 (linear regime: large m relative to n), starting l=2 from random throws away the linear-MSE floor, and by l=4 we are 3 noisy warm-starts removed from the truth.

Let me verify the simple recipe works on all 5 violations.


In [9]:
# Run the simple recipe on all 5 targets
print(f'{"config":<25} {"shallow_MSE":>12} {"stored_MSE":>12} {"warm_start":>12} {"final_MSE":>12} {"gain_vs_stored":>15}')
print('-' * 100)

simple_results = []
for _, t in targets.iterrows():
    n, m, l_deep, S = int(t.n), int(t.m), int(t.l), t.S
    l_shallow = int(t.shallow_l)
    
    # Try EACH shallower l (1, 2, ..., l_deep-1) — pick the best shallow as source
    best_floor = float('inf')
    best_shallow = None
    best_l_shallow = None
    for l_s in range(1, l_deep):
        sh = load_best_model(n, m, l_s, S)
        if sh is None: continue
        mse_s = eval_mse(sh, S)
        if mse_s < best_floor:
            best_floor = mse_s
            best_shallow = sh
            best_l_shallow = l_s
    
    # Embed with zero noise, train with standard cosine
    deep = embed_shallow_in_deep(best_shallow, l_deep, noise=0.0, seed=42)
    init_mse = eval_mse(deep, S)
    losses, eval_curve = train_and_track(
        deep, S, lambda step, ms: schedule_standard_cosine(step, ms),
        max_steps=15000, batch_size=4096)
    final_mse = eval_mse(deep, S)
    
    stored_mse = float(t.mse_current)
    gain = stored_mse / final_mse
    
    cfg = f'n={n} m={m} l={l_deep} S={S}'
    print(f'{cfg:<25} {best_floor:>12.6f} {stored_mse:>12.6f} {init_mse:>12.6f} {final_mse:>12.6f} {gain:>14.1f}x')
    
    simple_results.append(dict(
        n=n, m=m, l=l_deep, S=S, shallow_l=best_l_shallow,
        shallow_mse=best_floor, stored_mse=stored_mse,
        init_mse=init_mse, final_mse=final_mse,
        gain_vs_stored=gain,
        beats_shallow=final_mse < best_floor,
    ))

print()
sr = pd.DataFrame(simple_results)
print(sr.to_string())
print()
print(f'All beat current stored: {(sr.final_mse < sr.stored_mse).all()}')
print(f'All beat shallow floor: {(sr.final_mse < sr.shallow_mse).all()}')
print(f'Mean gain vs stored: {sr.gain_vs_stored.mean():.1f}x')


config                     shallow_MSE   stored_MSE   warm_start    final_MSE  gain_vs_stored
----------------------------------------------------------------------------------------------------


n=32 m=16 l=4 S=0.95          0.000389     0.004811     0.000389     0.000033          145.4x


n=16 m=8 l=4 S=0.95           0.000439     0.002246     0.000439     0.000042           52.9x


n=32 m=16 l=3 S=0.95          0.000389     0.001603     0.000389     0.000052           30.7x


n=32 m=16 l=4 S=0.9           0.001489     0.006990     0.001489     0.000178           39.3x


n=16 m=8 l=3 S=0.95           0.000439     0.001276     0.000439     0.000060           21.4x

    n   m  l     S  shallow_l  shallow_mse  stored_mse  init_mse  final_mse  gain_vs_stored  beats_shallow
0  32  16  4  0.95          1     0.000389    0.004811  0.000389   0.000033      145.387587           True
1  16   8  4  0.95          1     0.000439    0.002246  0.000439   0.000042       52.857811           True
2  32  16  3  0.95          1     0.000389    0.001603  0.000389   0.000052       30.690982           True
3  32  16  4  0.90          1     0.001489    0.006990  0.001489   0.000178       39.301931           True
4  16   8  3  0.95          1     0.000439    0.001276  0.000439   0.000060       21.434987           True

All beat current stored: True
All beat shallow floor: True
Mean gain vs stored: 57.9x
